In [25]:
import numpy as np
import matplotlib.pyplot as plt
from simple_pe.param_est import find_metric_and_eigendirections, calc_dx_bounds, pe
from simple_pe.waveforms import waveform
from pesummary.utils.array import Array
from pesummary.utils.samples_dict import SamplesDict

In [2]:
from simple_pe import io
f_low = 20
asd_data = {'H1': '/home/ben.patterson/projects/simple-pe/examples/zero-noise/aligo_O4high.txt',
            'L1': '/home/ben.patterson/projects/simple-pe/examples/zero-noise/aligo_O4high.txt',
            'V1': '/home/ben.patterson/projects/simple-pe/examples/zero-noise/avirgo_O4high_NEW.txt'}
psds = io.load_psd_from_file(
           {}, asd_data, 1/32, f_low, 2048,
       )
hm_psd = io.calculate_harmonic_mean_psd(psds)

/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pycbc/types/array.py:390: RuntimeWarning: divide by zero encountered in divide
  return self._data.__rtruediv__(other)


In [39]:
0.9889 + 0.1393**2

1.00830449

In [35]:
snr = 21.2484

x = {'chirp_mass': 53.7231, 'symmetric_mass_ratio': 0.2258, 'chi_align': 0.1890, 'chi_p2': 0.9793}
dx_directions = list(x.keys())

g = find_metric_and_eigendirections(
    x, dx_directions, snr, f_low, hm_psd, approximant="IMRPhenomXPHM",
    mins=None, maxs=None, tolerance=0.05, max_iter=2,
    ncpus=1, mute_multiprocessing=True,
    n_ecc_gen=6, mismatch=None
)

Calculating the metric | iteration 0 < 2| error 0.016 > 0.00086

chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06


Calculating the metric | iteration 1 < 2| error 0.0059 > 0.00086: 

chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06


Failed to achieve requested tolerance.  Requested: 0.00086achieved 0.0021: 


In [36]:
def create_metric_vectors(
    g, mass_ratio_last=True, eccentricity_last=True
):
    """
    Calculates orthogonal vectors of metric dx space.

    :param g: metric object
    :param mass_ratio_last: whether to prioritise mass ratio parameter last
    :param eccentricity_last: whether to prioritise eccentric parameter last
    :return par_ev_dict: orthogonal vectors
    """

    # Order list of parameter directions
    params = np.array(g.dx_directions.copy())
    final_params = []
    if eccentricity_last:
        final_params += ['ecc10', 'ecc10sqrd']
    if mass_ratio_last:
        final_params += ['mass_ratio', 'inverted_mass_ratio',
                         'symmetric_mass_ratio']
    for final_param in final_params:
        if final_param in params:
            params = np.delete(params, np.where(params == final_param))
            params = np.append(params, final_param)
    base_par_evs = np.identity(len(params))

    # Orthogonalise in dx space
    default_evs = [np.array(g.normalized_evecs()[param])
                   for param in params]
    dx_evs = np.matmul(np.linalg.inv(default_evs), base_par_evs)
    ev_list = dx_evs.T
    orth_ev_list = []
    for i in range(len(ev_list)):
        new_ev = ev_list[i].copy()
        for j in range(len(orth_ev_list)):
            norm = np.dot(orth_ev_list[j], orth_ev_list[j])
            new_ev -= np.dot(ev_list[i], orth_ev_list[j])*orth_ev_list[j]/norm
        orth_ev_list.append(new_ev)
    for i in range(len(orth_ev_list)):
        orth_ev_list[i] = orth_ev_list[i]/np.sqrt(np.dot(orth_ev_list[i],
                                                         orth_ev_list[i]))

    # Convert back to parameter space and set appropriate elements to zero
    # This avoids machine precision issues causing non-zero elements
    par_evs = np.matmul(default_evs, np.array(orth_ev_list).T)
    for i in range(len(par_evs)):
        for j in range(i+1, len(par_evs)):
            par_evs[:, i][j] = 0
    par_ev_dict = pe.SimplePESamples(SamplesDict(params, par_evs))

    return par_ev_dict

def calc_dx_bounds(
    current_point, dx_ind, g, orth_vectors, bound, maxs=None, mins=None
):
    """
    Calculates bounds along vector in dx space

    :param current_point: current_point in dx coordinates
    :param dx_ind: index of orth_vectors
    :param g: metric object
    :param orth_vectors: orthogonal vectors
    :param bound: maximum size of bounds
    :param maxs: dictionary of maximum values for physical parameters
    :param mins: dictionary of minimum values for physical parameters
    :return bounds: bounds
    """

    # Calculate unit change in physical parameters
    delta_dx = [0 for i in range(len(orth_vectors.keys()))]
    delta_dx[dx_ind] = 1
    delta_params_list = np.matmul(orth_vectors.samples, delta_dx)
    delta_params = pe.SimplePESamples(
        SamplesDict(orth_vectors.keys(),
                    [[param] for param in delta_params_list])
    )

    # Calculate current point in physical parameters
    base_point = current_point.copy()
    base_point[dx_ind] = 0
    current_delta_list = np.matmul(orth_vectors.samples, base_point)
    current_params = [[current_delta_list[i] + g.x[dx][0]]
                      for i, dx in enumerate(orth_vectors.keys())]
    current_params = pe.SimplePESamples(SamplesDict(orth_vectors.keys(),
                                                    current_params))

    # Calculate allowed bounds
    lower_alpha = waveform.check_physical(
        current_params, delta_params, -bound, maxs=maxs, mins=mins
    )
    chi_a_lower = current_params['chi_align'] - bound*lower_alpha*delta_params['chi_align']
    if 'chi_p' in current_params.keys():
        chi_p_lower = current_params['chi_p'] - bound*lower_alpha*delta_params['chi_p']
    else:
        chi_p_lower = np.sqrt(current_params['chi_p2'] - bound*lower_alpha*delta_params['chi_p2'])
    print(chi_a_lower, chi_p_lower, np.sqrt(chi_a_lower**2 + chi_p_lower**2))
    if lower_alpha < 1:
        lower_alpha = lower_alpha[0]
    upper_alpha = waveform.check_physical(
        current_params, delta_params, bound, maxs=maxs, mins=mins
    )
    chi_a_upper = current_params['chi_align'] + bound*upper_alpha*delta_params['chi_align']
    if 'chi_p' in current_params.keys():
        chi_p_upper = current_params['chi_p'] + bound*upper_alpha*delta_params['chi_p']
    else:
        chi_p_upper = np.sqrt(current_params['chi_p2'] + bound*upper_alpha*delta_params['chi_p2'])
    print(chi_a_upper, chi_p_upper, np.sqrt(chi_a_upper**2 + chi_p_upper**2))
    if upper_alpha < 1:
        upper_alpha = upper_alpha[0]
    bounds = [(-lower_alpha*bound, upper_alpha*bound)]

    return bounds

In [37]:
orth_vectors = create_metric_vectors(g)
bounds = calc_dx_bounds(
            [0,0,0,0], 3, g, orth_vectors, 100, maxs=None, mins=None
        )

[0.88741108] [0.43886397] [0.99]
[0.90329973] [0.001] [0.90330029]


In [20]:
bounds

[(-0.15980324291976564, 0.13348316621795486)]

In [18]:
orth_vectors

{'chirp_mass': Array([0.2298879 , 0.67762061, 0.16689498, 0.21132999]),
 'chi_align': Array([0.        , 0.08010636, 0.01075029, 0.0768441 ]),
 'chi_p': Array([ 0.        ,  0.        ,  1.18474715, -1.49082469]),
 'symmetric_mass_ratio': Array([0.        , 0.        , 0.        , 0.05204355])}

In [30]:
0.9-0.076844*0.1598

0.8877203288000001

In [31]:
0.2+1.4908*0.1598

0.43822984

In [32]:
0.8877**2 + 0.438**2

0.97985529